In [0]:

%sql

-- Create first fact table, the fact table from cart items

CREATE OR REPLACE TABLE retail_lakehouse_dr.gold.fact_cart_items
USING DELTA
AS

WITH ranked_carts AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY source, source_record_id
            ORDER BY extracted_at DESC, run_id DESC
        ) AS rn
    FROM retail_lakehouse_dr.silver.carts
),

selected_carts AS (
    SELECT *
    FROM ranked_carts
    WHERE rn = 1
),

ranked_cart_items AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY
                source,
                cart_source_record_id,
                run_id,
                item_index
            ORDER BY extracted_at DESC
        ) AS rn
    FROM retail_lakehouse_dr.silver.cart_items
),

selected_items AS (
    SELECT *
    FROM ranked_cart_items
    WHERE rn = 1
)

SELECT
    ci.cart_key,
    c.cart_id,
    ci.customer_id,
    ci.product_id,
    ci.item_index,
    ci.product_name,
    ci.price,
    ci.quantity,
    ci.total AS gross_amount,
    ci.discount_percentage,
    ci.discounted_total AS net_amount,
    ci.source,
    ci.cart_source_record_id,
    ci.run_id AS item_run_id,
    ci.extracted_at AS item_extracted_at,
    c.run_id AS cart_run_id,
    c.extracted_at AS cart_extracted_at
FROM selected_items ci
INNER JOIN selected_carts c
    ON  ci.source = c.source
    AND ci.cart_source_record_id = c.source_record_id
    AND ci.cart_key = c.cart_key
    AND ci.run_id = c.run_id;

num_affected_rows,num_inserted_rows


In [0]:
%sql

-- Create the report for product sales


-- CREATE OR REPLACE VIEW retail_lakehouse_dr.gold.vw_product_sales AS
-- SELECT
--     source,
--     product_id,
--     product_name,
--     COUNT(DISTINCT cart_key) AS distinct_carts,
--     SUM(quantity) AS units_sold,
--     ROUND(SUM(gross_amount), 2) AS gross_sales,
--     ROUND(SUM(net_amount), 2) AS net_sales,
--     ROUND(SUM(gross_amount - net_amount), 2) AS discount_amount
-- FROM retail_lakehouse_dr.gold.fact_cart_items
-- GROUP BY
--     source,
--     product_id,
--     product_name;


In [0]:
%sql


-- Create the report for sales by source


-- CREATE OR REPLACE VIEW retail_lakehouse_dr.gold.vw_sales_by_source AS
-- SELECT
--     source,
--     COUNT(DISTINCT cart_key) AS distinct_carts,
--     COUNT(*) AS item_rows,
--     SUM(quantity) AS units_sold,
--     ROUND(SUM(gross_amount), 2) AS gross_sales,
--     ROUND(SUM(net_amount), 2) AS net_sales,
--     ROUND(SUM(gross_amount - net_amount), 2) AS discount_amount,
--     ROUND(
--         SUM(net_amount) / NULLIF(COUNT(DISTINCT cart_key), 0),
--         2
--     ) AS avg_net_amount_per_cart
-- FROM retail_lakehouse_dr.gold.fact_cart_items
-- GROUP BY source;


In [0]:
%sql

-- Create one of the dimensional tables, the products dimension

-- CREATE OR REPLACE VIEW retail_lakehouse_dr.gold.vw_dim_products AS

-- WITH ranked_products AS (
--     SELECT
--         *,
--         ROW_NUMBER() OVER (
--             PARTITION BY product_id
--             ORDER BY extracted_at DESC, run_id DESC
--         ) AS rn
--     FROM retail_lakehouse_dr.silver.products
-- )

-- SELECT
--     product_id,
--     product_name,
--     category,
--     brand,
--     price,
--     discount_percentage,
--     rating,
--     stock,
--     sku,
--     availability_status,
--     minimum_order_quantity,
--     extracted_at AS product_extracted_at
-- FROM ranked_products
-- WHERE rn = 1;


In [0]:
%sql

-- We will enrich the sales insights using the products dimension table

-- CREATE OR REPLACE VIEW retail_lakehouse_dr.gold.vw_sales_enriched AS
-- SELECT
--     f.cart_key,
--     f.cart_id,
--     f.customer_id,
--     f.product_id,
--     f.item_index,
--     f.product_name,
--     p.category,
--     p.brand,
--     p.sku,
--     f.price AS sale_price,
--     p.price AS current_product_price,
--     f.quantity,
--     f.gross_amount,
--     f.discount_percentage AS sale_discount_percentage,
--     f.net_amount,
--     f.gross_amount - f.net_amount AS discount_amount,
--     p.rating AS current_product_rating,
--     p.stock AS current_stock,
--     p.availability_status AS current_availability_status,
--     f.source,
--     f.cart_source_record_id,
--     f.item_run_id,
--     f.item_extracted_at,
--     f.cart_run_id,
--     f.cart_extracted_at,
--     p.product_extracted_at
-- FROM retail_lakehouse_dr.gold.fact_cart_items f
-- LEFT JOIN retail_lakehouse_dr.gold.vw_dim_products p
--     ON f.product_id = p.product_id;


In [0]:
%sql

--  Create the report for sales by category

-- CREATE OR REPLACE VIEW retail_lakehouse_dr.gold.vw_sales_by_category AS
-- SELECT
--     source,
--     category,
--     COUNT(DISTINCT cart_key) AS distinct_carts,
--     COUNT(*) AS item_rows,
--     SUM(quantity) AS units_sold,
--     ROUND(SUM(gross_amount), 2) AS gross_sales,
--     ROUND(SUM(net_amount), 2) AS net_sales,
--     ROUND(SUM(discount_amount), 2) AS discount_amount,
--     ROUND(
--         100.0 * SUM(discount_amount)
--         / NULLIF(SUM(gross_amount), 0),
--         2
--     ) AS discount_rate_pct
-- FROM retail_lakehouse_dr.gold.vw_sales_enriched
-- GROUP BY
--     source,
--     category;
